# Simple: Split Fields with PAMOLA.CORE

**Goal**: Learn field splitting basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Apply field-based splitting using execute()
- Compare before/after results
- Understand column grouping strategies

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── transformations/
        └── 01_split_fields_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd 
import numpy as np 
import sys 
import os 
import json 
from pathlib import Path 
from datetime import datetime 
from IPython.display import Image, display 
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.transformations.splitting.split_fields_op import SplitFieldsOperation 
    from pamola_core.utils.ops.op_data_source import DataSource 
    from pamola_core.utils.progress import HierarchicalProgressTracker 
    from pamola_core.utils.tasks.task_reporting import TaskReporter 
    from pamola_core.io.csv import read_csv 
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Run to load data from `examples/data_examples/sample.csv`
- Auto-creates sample data if file missing
- Review dataset structure before proceeding

**What you'll see:**
- Dataset summary (20 records, 14 columns)
- First 5 employee records
- Column statistics (types, unique value counts)
- Field categories: ID, personal, work, and contact information

**Note:** Sample data contains employee records with mixed field types suitable for splitting

In [ ]:
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'

if not data_path.exists():
    print("⚠️  File not found, creating sample data...")
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample employee data with mixed field types
    np.random.seed(42)
    sample_data = pd.DataFrame({
        # ID field
        'employee_id': [f'E{str(i).zfill(4)}' for i in range(1, 21)],
        
        # Personal information
        'first_name': ['Alice', 'Bob', 'Carol', 'David', 'Eve', 'Frank', 'Grace', 'Henry', 'Iris', 'Jack',
                      'Karen', 'Leo', 'Mary', 'Nathan', 'Olivia', 'Paul', 'Quinn', 'Rachel', 'Sam', 'Tina'],
        'last_name': ['Johnson', 'Smith', 'Williams', 'Brown', 'Davis', 'Miller', 'Wilson', 'Moore', 'Taylor', 'Anderson',
                     'Thomas', 'Jackson', 'White', 'Harris', 'Martin', 'Thompson', 'Garcia', 'Martinez', 'Robinson', 'Clark'],
        'birth_date': pd.date_range('1980-01-01', periods=20, freq='180D'),
        'gender': np.random.choice(['Male', 'Female'], 20),
        
        # Work information
        'department': np.random.choice(['IT', 'HR', 'Sales', 'Finance', 'Marketing'], 20),
        'job_title': np.random.choice(['Manager', 'Analyst', 'Specialist', 'Coordinator', 'Director'], 20),
        'salary': np.random.randint(50000, 150000, 20),
        'hire_date': pd.date_range('2015-01-01', periods=20, freq='90D'),
        'years_experience': np.random.randint(1, 20, 20),
        
        # Contact information
        'email': [f'{name.lower()}.{surname.lower()}@company.com' 
                 for name, surname in zip(
                     ['alice', 'bob', 'carol', 'david', 'eve', 'frank', 'grace', 'henry', 'iris', 'jack',
                      'karen', 'leo', 'mary', 'nathan', 'olivia', 'paul', 'quinn', 'rachel', 'sam', 'tina'],
                     ['johnson', 'smith', 'williams', 'brown', 'davis', 'miller', 'wilson', 'moore', 'taylor', 'anderson',
                      'thomas', 'jackson', 'white', 'harris', 'martin', 'thompson', 'garcia', 'martinez', 'robinson', 'clark']
                 )],
        'phone': [f'+1-555-{np.random.randint(100, 999)}-{np.random.randint(1000, 9999)}' for _ in range(20)],
        'address': [f'{np.random.randint(100, 9999)} Main St' for _ in range(20)],
        'city': np.random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix'], 20)
    })
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created")

df = read_csv(data_path)
print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:20s} ({dtype_str:10s})")

## Step 3: Setup Task Environment

**How to use:**
1. **CUSTOMIZE id_field and field_groups** in `create_config_kwargs()`
   - Default id_field: `"resume_id"`
   - Default groups: `{"personal_info": ['first_name', 'last_name'], "company_info": ['company_current', 'job_title_current']}`
   - Change to YOUR dataset's ID column and field groupings
2. Run to validate fields and setup environment

**What you'll see:**
- ID field validation (will be included in all groups)
- Field group validation (group count, fields per group)
- Task directory created (✓)
- TaskReporter, DataSource, progress tracker ready (✓)

**Note:** ID field and all group fields must exist in dataset

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "id_field": "resume_id",
        "field_groups": {
            "personal_info": ['first_name', 'last_name'],
            "company_info": ['company_current', 'job_title_current']
        }
    }

kwargs = create_config_kwargs()
id_field = kwargs["id_field"]
field_groups = kwargs["field_groups"]

# Validate ID field and groups
print(f"\n🔍 Validating field configuration...\n")
if id_field not in df.columns:
    raise ValueError(
        f"❌ ID field '{id_field}' not found in dataset!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'id_field' in create_config_kwargs()"
    )

print(f"✓ ID field: '{id_field}'")
print(f"  Will be included in each group\n")

print(f"✓ Field groups: {len(field_groups)}")
for group_name, fields in field_groups.items():
    # Validate all fields exist
    missing_fields = [f for f in fields if f not in df.columns]
    if missing_fields:
        raise ValueError(
            f"❌ Fields {missing_fields} in group '{group_name}' not found in dataset!\n"
            f"Available columns: {', '.join(df.columns)}"
        )
    print(f"  {group_name}: {len(fields)} fields")
    print(f"    → {', '.join(fields)}")

# Create task directory
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_001",
    task_type="transformation",
    description="Split data by field groups",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(dataframes={"main_dataset": df})
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Splitting fields into {len(field_groups)} groups",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Review configuration parameters
- Run to execute field splitting
- Monitor progress bar (6 steps: validate → split → save → metrics → visualize → complete)

**Key parameters:**
- `id_field`: Identifier field to include in all groups
- `field_groups`: Dictionary mapping group names to field lists
- `include_id_field=True`: Include ID in each output
- `generate_visualization=True`: Create field distribution charts
- `save_output=True`: Save split files

**What you'll see:**
- Configuration summary with all parameters
- Progress bar: validate → split → save → metrics → visualize → complete
- Completion status: "completed" (verify no errors)
- Artifact count (4-6 files expected)

**Note:** Each group will include the ID field plus specified fields

In [ ]:
# Create operation with production-style configuration
operation = SplitFieldsOperation(
    # Core parameters
    id_field=id_field,
    field_groups=field_groups,
    include_id_field=True,  # Include ID in each group
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Operation configured")
print(f"  ID field:         {operation.id_field}")
print(f"  Field groups:     {len(operation.field_groups)}")
print(f"  Include ID:       {operation.include_id_field}")
print(f"  Visualizations:   {operation.generate_visualization}")
print(f"  Save output:      {operation.save_output}")
print(f"  Force recalc:     {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing split fields...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Run to load and compare split results
- Review field distribution across groups
- Verify ID field included in all groups

**What you'll see:**
- Generated artifacts list (output, metrics, visualizations)
- Split summary table (records and field count per group)
- Field distribution showing which columns in each group
- Sample records from each group (first 3 rows)
- Key metrics: execution time, field counts, group statistics

**Note:** All groups should have same record count (20) but different field counts

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load output files
output_files = list(task_dir.glob('output/*.csv'))
if output_files:
    print("\n" + "=" * 80)
    print("📊 SPLIT RESULTS")
    print("=" * 80)
    
    # Load all split files
    split_dfs = {}
    for output_file in output_files:
        # Extract group name from filename
        parts = output_file.stem.split('_')
        try:
            output_idx = parts.index('output')
            group_name = '_'.join(parts[4:output_idx])  # After 'operation'
        except (ValueError, IndexError):
            group_name = output_file.stem
        
        split_dfs[group_name] = pd.read_csv(output_file)
    
    # Show summary
    print(f"\n📈 Split Summary:")
    print(f"{'Group':<20} {'Records':>10} {'Fields':>10}")
    print("-" * 45)
    for group_name, group_df in sorted(split_dfs.items()):
        records = len(group_df)
        fields = len(group_df.columns)
        print(f"{group_name:<20} {records:>10} {fields:>10}")
    
    # Show field distribution
    print(f"\n📋 Field Distribution:")
    for group_name, group_df in sorted(split_dfs.items()):
        print(f"\n  {group_name}:")
        print(f"    {', '.join(group_df.columns)}")
    
    # Show sample from each group
    print(f"\n🔍 Sample Records from Each Group:")
    for group_name, group_df in sorted(split_dfs.items()):
        print(f"\n  {group_name} (showing first 3):")
        print(group_df.head(3).to_string(index=False))
    
    print("\n" + "=" * 80)
    print("✨ SUMMARY")
    print("=" * 80)
    print(f"  Original dataset:   {len(df)} records, {len(df.columns)} fields")
    print(f"  Number of groups:   {len(split_dfs)}")
    print(f"  ID field included:  Yes ({id_field} in all groups)")
    
    # Display result metrics
    if result.metrics:
        print("\n📊 Key Metrics:")
        for key, value in list(result.metrics.items())[:10]:
            if isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:30s}: {value:.4f}")
                else:
                    print(f"  {key:30s}: {value}")
    
    print(f"\n💡 Fields successfully split into {len(split_dfs)} logical groups!")
else:
    print("⚠️  No output files found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Run to display all generated files
- Navigate to directories for manual inspection
- Verify each folder has expected file count

**What you'll see:**
- Directory structure tree (output/, metrics/, visualizations/, config/)
- File count per directory (2 files in output/ for 2 groups)
- File names with sizes in KB
- Absolute path to task directory for manual navigation

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Split CSV files (one per field group)
├── metrics/          # Field distribution metrics
├── visualizations/   # Field distribution charts
└── config/           # Operation config
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

for subdir in ['output', 'metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")

print("\n✅ All artifacts saved successfully!")

## Step 7: Detailed Artifact Review

**How to use:**
- Review all generated artifacts in detail
- Automatically loads the NEWEST files from each category
- Excludes system files (like data_types metrics)
- Displays visualizations inline for easy review

**What you'll see:**
1. **Metrics JSON**: Split statistics and field distribution
2. **Output Data**: Split records with field lists from each group
3. **Visualizations**: Charts showing field distribution

**Note:** Only the most recent files are shown to avoid confusion from multiple runs

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. METRICS JSON (NEWEST - with filtering)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / "metrics"
if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")
else:
    metrics_files = list(metrics_dir.glob("*.json"))

    if not metrics_files:
        print("⚠️  No metrics files found")
    else:
        # 1) Exclude data_types files
        filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

        if filtered:
            target_files = filtered
            print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
        else:
            target_files = metrics_files
            print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

        # 2) Pick latest
        target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = target_files[0]

        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
        print("=" * 80)

        try:
            with open(latest_metrics_file, "r", encoding="utf-8") as f:
                data = json.load(f)

            metadata = data.get("metadata", {})
            metrics = data.get("metrics", {})

            # ------------------------------------------------------------------
            # METADATA
            # ------------------------------------------------------------------
            print("📋 METADATA:")
            print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
            print(f"   Name: {metadata.get('name', 'N/A')}")

            if "operation" in metadata:
                op = metadata["operation"]
                print(f"   Operation: {op.get('class', 'N/A')}.{op.get('function', 'N/A')}")

            # ------------------------------------------------------------------
            # DATASET METRICS
            # ------------------------------------------------------------------
            print("\n📊 DATASET METRICS:")

            dataset_metrics = [
                ("total_input_records", "Input Records"),
                ("total_output_records", "Output Records"),
                ("total_input_fields", "Input Fields"),
                ("total_output_fields", "Output Fields"),
                ("id_field", "ID Field"),
                ("number_of_splits", "Number of Splits"),
            ]

            for key, label in dataset_metrics:
                if key in metrics:
                    print(f"   {label:<22}: {metrics[key]}")

            # ------------------------------------------------------------------
            # SPLIT INFORMATION
            # ------------------------------------------------------------------
            if "split_info" in metrics:
                print("\n📊 SPLIT INFORMATION:")
                split_info = metrics["split_info"]

                print(f"{'Group':<20} {'Field Count':>15}")
                print("-" * 40)

                for group_name, info in split_info.items():
                    field_count = info.get("field_count", 0)
                    print(f"{group_name:<20} {field_count:>15}")

                # --------------------------------------------------------------
                # FIELDS PER GROUP
                # --------------------------------------------------------------
                print("\n📋 FIELDS PER GROUP:")
                for group_name, info in split_info.items():
                    fields = info.get("included_fields", [])

                    print(f"\n   {group_name}:")
                    if fields:
                        print(f"      {', '.join(fields)}")
                    else:
                        print("      (no fields listed)")

            # ------------------------------------------------------------------
            # PERFORMANCE
            # ------------------------------------------------------------------
            print("\n⚡ PERFORMANCE:")

            if "execution_time_seconds" in metrics:
                print(
                    f"   Execution Time        : "
                    f"{metrics['execution_time_seconds']:.4f}s"
                )

            if "processing_date" in metrics:
                print(f"   Processing Date       : {metrics['processing_date']}")

        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. OUTPUT DATA (ALL SPLIT FILES)
print("\n\n2️⃣ OUTPUT DATA (Split Files):")
print("-" * 80)

output_dir = task_dir / 'output'
if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")
else:
    output_files = list(output_dir.glob('*.csv'))
    
    if not output_files:
        print("⚠️  No output files found")
    else:
        # Sort by modification time (newest first)
        output_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # Get timestamp from latest file to identify the batch
        latest_time = output_files[0].stat().st_mtime
        time_threshold = 10  # seconds
        latest_batch = [
            f for f in output_files 
            if abs(f.stat().st_mtime - latest_time) < time_threshold
        ]
        
        # Show info
        latest_mtime = datetime.fromtimestamp(latest_time)
        print(f"✓ Found {len(output_files)} total output file(s)")
        print(f"📄 Showing LATEST batch: {len(latest_batch)} files")
        print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
        
        # Display each split file
        for i, output_file in enumerate(sorted(latest_batch, key=lambda x: x.name), 1):
            # Extract group name
            parts = output_file.stem.split('_')
            try:
                output_idx = parts.index('output')
                group_name = '_'.join(parts[4:output_idx])
            except (ValueError, IndexError):
                group_name = output_file.stem
            
            print(f"\n{i}. {group_name}")
            print(f"   File: {output_file.name}")
            print(f"   Size: {output_file.stat().st_size / 1024:.1f} KB")
            print("-" * 60)
            
            try:
                split_df = pd.read_csv(output_file)
                print(f"   Shape: {split_df.shape[0]} rows × {split_df.shape[1]} columns")
                
                # Show fields in this split
                print(f"   Fields: {', '.join(split_df.columns)}")
                
                # Show first 3 rows
                print(f"\n   First 3 rows:")
                print(split_df.head(3).to_string(index=False, max_colwidth=20))
            
            except Exception as e:
                print(f"   ❌ Error reading file: {e}")
        
        # Show info about older files if any
        if len(output_files) > len(latest_batch):
            print(f"\n💡 Note: {len(output_files) - len(latest_batch)} older file(s) not shown")

# 3. VISUALIZATIONS (NEWEST BATCH)
print("\n\n3️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        # Sort by modification time (newest first)
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # Get timestamp from latest file to identify the batch
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            # Group files from same operation (within 10 seconds)
            time_threshold = 10  # seconds
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            # Sort batch by name for consistent ordering
            latest_viz_batch.sort(key=lambda x: x.name)
            
            # Show info
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            # Display each visualization from latest batch
            for i, viz_file in enumerate(latest_viz_batch, 1):
                # Extract readable name from filename
                viz_name = viz_file.stem.replace('_', ' ').title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        # Show info about older visualizations if any
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**  
✅ Load data from examples/data_examples/  
✅ Setup environment with TaskReporter, DataSource, ProgressTracker  
✅ Configure split operation with field groups  
✅ Execute using operation.execute()  
✅ Analyze results and review artifacts  

**Key takeaways:**
- Field groups organize columns by logical categories
- ID field automatically included in all groups
- Each group saved as separate CSV file
- Visualizations show field distribution
- All artifacts saved in structured directories

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_split_fields_advanced.ipynb`** includes:
- Dynamic field group generation
- Conditional field inclusion
- Field dependency management
- Schema validation and enforcement
- Integration with data governance
- Complex field hierarchies

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)